In [1]:
import pandas as pd
import polars as pl
# 2. Define the output file path
complete_path = "../../data/data_hospitales/sistema_salud_egresos_limpio.parquet"

# 3. Write the Polars DataFrame to a Parquet file
base_general= pl.read_parquet(complete_path)


In [ ]:
new_cords_parroquia_dict = {
    ("PICHINCHA", "QUITO", "QUITO , CABECERA CANTONAL Y CAPITAL PROVINCIAL"): (-0.19506169985974994, -78.49475894065186),
    ("GUAYAS", "GUAYAQUIL", "GUAYAQUIL , CABECERA CANTONAL Y CAPITAL PROVINCIAL"): (-2.1873106383919407, -79.89663942964654),
    ("GUAYAS", "GUAYAQUIL", "TARQUI"): (-2.129950856875764, -79.89878924477644)
}

def aplicar_nuevas_coordenadas(df, sufijo):
    col_prov = f"prov_{sufijo}"
    col_cant = f"cant_{sufijo}"
    col_parr = f"parr_{sufijo}"
    col_lat = f"lat_{sufijo}"
    col_lon = f"lon_{sufijo}"

    for (p, c, par), (new_lat, new_lon) in new_cords_parroquia_dict.items():
        # Definimos la condición de coincidencia exacta
        condicion = (
            (pl.col(col_prov) == p) & 
            (pl.col(col_cant) == c) & 
            (pl.col(col_parr) == par)
        )
        
        # Aplicamos el cambio a latitud y longitud
        df = df.with_columns([
            pl.when(condicion).then(pl.lit(new_lat)).otherwise(pl.col(col_lat)).alias(col_lat),
            pl.when(condicion).then(pl.lit(new_lon)).otherwise(pl.col(col_lon)).alias(col_lon)
        ])
    
    return df

# Aplicar a residencia y ubicación
base_general_filtrada = aplicar_nuevas_coordenadas(base_general_filtrada, "res")
base_general_filtrada = aplicar_nuevas_coordenadas(base_general_filtrada, "ubi")


In [2]:
print("columnas_originales:", base_general.columns)

columnas_originales: ['prov_ubi', 'cant_ubi', 'parr_ubi', 'area_ubi', 'clase', 'tipo', 'entidad', 'sector', 'mes_inv', 'nac_pac', 'nom_pais', 'cod_pais', 'sexo', 'cod_edad', 'edad', 'etnia', 'prov_res', 'cant_res', 'parr_res', 'area_res', 'anio_ingr', 'mes_ingr', 'dia_ingr', 'fecha_ingr', 'anio_egr', 'mes_egr', 'dia_egr', 'fecha_egr', 'dia_estad', 'con_egrpa', 'esp_egrpa', 'cau_cie10', 'causa3', 'cap221rx', 'cau221rx', 'cau298rx', 'archivo_origen', 'tipo_seg', 'dis_pac', 'edad_std', 'cau221rx_std', 'cau_cie10_std', 'lat_res', 'lon_res', 'lat_ubi', 'lon_ubi', 'code_parr_res', 'code_parr_ubi', 'code_cant_res', 'code_cant_ubi', 'sindrome_metabolico']


In [3]:
identificadores = [
    'clase',
    'tipo',
    'entidad',
    'sector'
]
territorio_ubi=[    
    'prov_ubi',
    'cant_ubi',
    'parr_ubi',
    #'area_ubi',

]
territorio_res = [
    'prov_res',
    'cant_res',
    'parr_res',
    #'area_res',
]

fecha_egr = [
    'fecha_egr',

]

estadisticas = [
    'dia_estad', #dias de estadia
    'con_egrpa', # egreso vivo o muerto (antes o despues de 48 horas)
]

diagnostico=[
    "cau_cie10",
    "causa3",
    "cap221rx",
    "cau221rx",
    "cau298rx",
]

demografia=[
    "sector",
    "mes_inv",
    "nac_pac",
    "nom_pais",
    "cod_pais",
    "sexo",
    "cod_edad",
    "edad",
    "etnia",
    
]


In [4]:
import pandas as pd
import polars as pl
from keplergl import KeplerGl

In [5]:
# Filtrar solo síndrome metabólico y columnas necesarias para flujos
DATE_FORMAT = "%Y-%m-%d"

sm = (
    base_general
    .filter(pl.col("sindrome_metabolico") == True)
    .select([
        *identificadores,  # ['clase', 'tipo', 'entidad', 'sector']
        "parr_res", "parr_ubi",
        "code_parr_res", "code_parr_ubi",
        "lat_res", "lon_res",
        "lat_ubi", "lon_ubi",
        "cau221rx_std","cau_cie10_std",
        "fecha_egr",
        "anio_egr",
    ])
    .with_columns(
        pl.col("fecha_egr")
          .str.to_date(DATE_FORMAT)
          .dt.truncate("1mo")
          .alias("fecha_egr_mes")
    )
    .drop_nulls(subset=["lat_res", "lon_res", "lat_ubi", "lon_ubi", "fecha_egr_mes"])
)

print(f"Registros síndrome metabólico: {sm.shape[0]:,}")
print(sm.head(3))

Registros síndrome metabólico: 298,258
shape: (3, 17)
┌────────────┬────────────┬───────────┬─────────┬───┬───────────┬───────────┬──────────┬───────────┐
│ clase      ┆ tipo       ┆ entidad   ┆ sector  ┆ … ┆ cau_cie10 ┆ fecha_egr ┆ anio_egr ┆ fecha_egr │
│ ---        ┆ ---        ┆ ---       ┆ ---     ┆   ┆ _std      ┆ ---       ┆ ---      ┆ _mes      │
│ str        ┆ str        ┆ str       ┆ str     ┆   ┆ ---       ┆ str       ┆ str      ┆ ---       │
│            ┆            ┆           ┆         ┆   ┆ str       ┆           ┆          ┆ date      │
╞════════════╪════════════╪═══════════╪═════════╪═══╪═══════════╪═══════════╪══════════╪═══════════╡
│ HOSPITAL   ┆ Agudo      ┆ MINISTERI ┆ Público ┆ … ┆ E115      ┆ 2015-04-1 ┆ 2015.0   ┆ 2015-04-0 │
│ GENERAL    ┆            ┆ O DE      ┆         ┆   ┆ DIABETES  ┆ 0         ┆          ┆ 1         │
│            ┆            ┆ SALUD     ┆         ┆   ┆ MELLITUS  ┆           ┆          ┆           │
│            ┆            ┆ PUBLICA  

In [6]:
# Agrupar por parroquia de residencia → parroquia del hospital + mes + año de egreso
flujos_sm = (
    sm
    .group_by([
        *identificadores,  # ['clase', 'tipo', 'entidad', 'sector']
        "cau221rx_std","cau_cie10_std",
        "code_parr_res", "parr_res",
        "code_parr_ubi", "parr_ubi",
        "fecha_egr_mes", "anio_egr",
    ])
    .agg([
        pl.col("lat_res").mean().alias("lat_origen"),
        pl.col("lon_res").mean().alias("lon_origen"),
        pl.col("lat_ubi").mean().alias("lat_destino"),
        pl.col("lon_ubi").mean().alias("lon_destino"),
        pl.len().alias("conteo"),
    ])
    .sort(["fecha_egr_mes", "conteo"], descending=[False, True])
    .to_pandas()
)

# Conversión a timestamp — mismo enfoque que funcionaba antes
flujos_sm["fecha_egr_mes"] = flujos_sm["fecha_egr_mes"].dt.to_period("M").dt.to_timestamp()

print(f"Flujos únicos: {len(flujos_sm):,}")
print(f"Tipo fecha_egr_mes: {flujos_sm['fecha_egr_mes'].dtype}")
print(f"Años disponibles: {sorted(flujos_sm['anio_egr'].unique())}")

# Convertir a milisegundos (Unix Timestamp)
# Convertir a string con formato ISO completo
flujos_sm["fecha_egr_mes"] = flujos_sm["fecha_egr_mes"].dt.strftime('%Y-%m-%dT%H:%M:%S')

print(f"Tipo: {flujos_sm['fecha_egr_mes'].dtype}") # Saldrá 'object'
print(flujos_sm["fecha_egr_mes"].iloc[0]) # Verás '2015-04-01T00:00:00'
# Debería mostrar int64
flujos_sm.head(5)

Flujos únicos: 216,943
Tipo fecha_egr_mes: datetime64[ns]
Años disponibles: ['2015.0', '2016.0', '2017.0', '2018.0', '2019.0', '2020.0', '2021.0', '2022.0', '2023.0', '2024.0']
Tipo: object
2015-01-01T00:00:00


,clase,tipo,entidad,sector,cau221rx_std,cau_cie10_std,code_parr_res,parr_res,code_parr_ubi,parr_ubi,fecha_egr_mes,anio_egr,lat_origen,lon_origen,lat_destino,lon_destino,conteo
0,HOSPITAL BASICO,Sin tipo hospitales básicos,INSTITUTO ECUATORIANO DE SEGURIDAD SOCIAL,Público,Diabetes mellitus (E10-E14),"E149 DIABETES MELLITUS NO ESPECIFICADA, SIN ...",EC070150,"MACHALA , CABECERA CANTONAL Y CAPITAL PROVINCIAL",EC070150,"MACHALA , CABECERA CANTONAL Y CAPITAL PROVINCIAL",2015-01-01T00:00:00,2015.0,-3.264001,-79.962452,-3.264001,-79.962452,76
1,HOSPITAL DE ESPECIALIDADES,Agudo,INSTITUTO ECUATORIANO DE SEGURIDAD SOCIAL,Público,Diabetes mellitus (E10-E14),"E149 DIABETES MELLITUS NO ESPECIFICADA, SIN ...",EC170150,"QUITO , CABECERA CANTONAL Y CAPITAL PROVINCIAL",EC170150,"QUITO , CABECERA CANTONAL Y CAPITAL PROVINCIAL",2015-01-01T00:00:00,2015.0,-0.195062,-78.494759,-0.195062,-78.494759,20
2,HOSPITAL BASICO,Sin tipo hospitales básicos,INSTITUTO ECUATORIANO DE SEGURIDAD SOCIAL,Público,Diabetes mellitus (E10-E14),"E149 DIABETES MELLITUS NO ESPECIFICADA, SIN ...",EC070101,LA PROVIDENCIA,EC070150,"MACHALA , CABECERA CANTONAL Y CAPITAL PROVINCIAL",2015-01-01T00:00:00,2015.0,-3.268051,-79.902517,-3.264001,-79.962452,18
3,HOSPITAL DE ESPECIALIDADES,Agudo,MINISTERIO DE SALUD PUBLICA,Público,Diabetes mellitus (E10-E14),"E112 DIABETES MELLITUS NO INSULINODEPENDIENTE,...",EC170150,"QUITO , CABECERA CANTONAL Y CAPITAL PROVINCIAL",EC170113,ITCHIMBIA,2015-01-01T00:00:00,2015.0,-0.195062,-78.494759,-0.210864,-78.477875,14
4,HOSPITAL DE ESPECIALIDADES,Agudo,INSTITUTO ECUATORIANO DE SEGURIDAD SOCIAL,Público,Obesidad y otros tipos de hiperalimentación (E...,"E669 OBESIDAD, NO ESPECIFICADA",EC090112,TARQUI,EC090150,"GUAYAQUIL , CABECERA CANTONAL Y CAPITAL PROVIN...",2015-01-01T00:00:00,2015.0,-2.280790,-80.102694,-2.187311,-79.896639,13


In [7]:
flujos_sm["cau221rx_std"].value_counts()

cau221rx_std
Diabetes mellitus (E10-E14)                                                         118984
Enfermedades hipertensivas (I10-I15)                                                 51110
Enfermedades isquémicas del corazón (I20-I25)                                        22872
Obesidad y otros tipos de hiperalimentación (E65-E68)                                13077
Otros trastornos maternos relacionados principalmente con el  embarazo (O20-O29)      9591
Trastornos metabólicos (E70-E90)                                                      1309
Name: count, dtype: int64

In [21]:
%run hex_config_sm.py

config_new=config_sm

In [22]:
mapa = KeplerGl(height=800,config=config_new)
mapa.add_data(data=flujos_sm, name="prospeccion_base")
mapa

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(config={'version': 'v1', 'config': {'visState': {'filters': [{'dataId': ['prospeccion_base'], 'id': '…

In [ ]:
# Opcional: exportar el mapa a HTML
with open('hex_config_sm.py', 'w') as f:
   f.write('config_sm = {}'.format(mapa.config))


In [ ]:
   
mapa.save_to_html(file_name="flujos_sindrome_metabolico.html")